# Fix ASR Transcriptions (batch_size=100)

Re-runs Whisper at batch_size=100 on all runs where the stored `asr_transcription`
produces a different set_overlap score than what is stored in `run_summary.json`.
Updates the JSON in-place with the correct transcription.

Run this on Colab with a GPU runtime.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo and install dependencies
import os
os.chdir('/content')
!git clone https://github.com/Vorgesetzter/StyleTTS2.git
os.chdir('/content/StyleTTS2')
!pip install -q styletts2 whisper-openai nltk torchaudio

In [ ]:
# Download outputs from GCS
GCS_BUCKET = 'thesis-data-2026'
!gsutil -m cp -r gs://{GCS_BUCKET}/outputs/results /content/StyleTTS2/outputs/

In [ ]:
import re, json, os, sys
sys.path.insert(0, '/content/StyleTTS2')
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import torch
import torchaudio
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

STOPWORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

def content_lemmas(text):
    clean = re.sub(r'[^\w\s]', '', text.lower())
    words = set(clean.split()) - STOPWORDS
    result = set()
    for w in words:
        for pos in ('a', 'v', 'n', 'r'):
            lemma = LEMMATIZER.lemmatize(w, pos=pos)
            if lemma != w:
                result.add(lemma)
                break
        else:
            result.add(w)
    return result

def set_overlap(gt, asr):
    gt_l = content_lemmas(gt)
    return len(gt_l & content_lemmas(asr)) / len(gt_l) if gt_l else 1.0

def load_audio(path, target_sr=24000):
    waveform, sr = torchaudio.load(path)
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, target_sr)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    return waveform  # (1, samples)

print('Helpers loaded OK')

In [ ]:
# Collect all runs with score/transcription mismatch
ROOTS = [
    '/content/StyleTTS2/outputs/results/TTS',
    '/content/StyleTTS2/outputs/results/Waveform',
]

mismatches = []
for root in ROOTS:
    if not os.path.isdir(root):
        print(f'[SKIP] {root} not found')
        continue
    for sent in sorted(os.listdir(root)):
        sent_path = os.path.join(root, sent)
        if not sent.startswith('sentence_') or not os.path.isdir(sent_path):
            continue
        for run in sorted(os.listdir(sent_path)):
            run_path = os.path.join(sent_path, run)
            sp = os.path.join(run_path, 'run_summary.json')
            wp = os.path.join(run_path, 'best_mixed.wav')
            if not os.path.exists(sp) or not os.path.exists(wp):
                continue
            with open(sp) as f:
                s = json.load(f)
            gt = s['text_data']['ground_truth_text']
            asr = s['text_data']['asr_transcription']
            stored = s['success_metrics']['fitness_scores'].get('SET_OVERLAP')
            if stored is None:
                continue
            recomp = set_overlap(gt, asr)
            if abs(recomp - stored) > 0.05:
                mismatches.append({
                    'summary_path': sp,
                    'wav': wp,
                    'gt': gt,
                    'old_asr': asr,
                    'stored_score': stored,
                })

print(f'Runs with mismatch: {len(mismatches)}')

In [ ]:
from Models.whisper import Whisper

BATCH_SIZE = 100
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
asr_model = Whisper(device=device)

correct = still_wrong = updated = 0

for m in mismatches:
    audio = load_audio(m['wav']).to(device).expand(BATCH_SIZE, -1)
    texts, _ = asr_model.inference(audio)
    new_asr = texts[0]
    new_score = set_overlap(m['gt'], new_asr)
    ok = abs(new_score - m['stored_score']) <= 0.05

    if ok:
        correct += 1
        # Update the JSON with the corrected transcription
        with open(m['summary_path']) as f:
            s = json.load(f)
        s['text_data']['asr_transcription'] = new_asr
        with open(m['summary_path'], 'w') as f:
            json.dump(s, f, indent=2)
        updated += 1
    else:
        still_wrong += 1

    status = 'OK     ' if ok else 'WRONG  '
    print(f'[{status}] stored={m["stored_score"]:.3f}  new={new_score:.3f}')
    print(f'  old: {m["old_asr"]}')
    print(f'  new: {new_asr}')

print(f'\nNow correct:  {correct}/{len(mismatches)}')
print(f'Still wrong:  {still_wrong}/{len(mismatches)}')
print(f'JSONs updated: {updated}')

In [ ]:
# Upload corrected JSONs back to GCS
!gsutil -m cp -r /content/StyleTTS2/outputs/results gs://{GCS_BUCKET}/outputs/